# 13 — Model 5: XAI + CMI (transformer, top rung) + attention

Top rung of the faithfulness phase, and the **only rung with a fourth attribution method: attention weights**
— which is why patch 60 was chosen (one token per 60-sample region, so attention maps onto the CMI grid with
no aggregation). Same pipeline as the CNN rungs, inline for the methods chapter, plus §5 (attention). Model 5
is the **transformer (patch 60 — the ladder rung, not patch 15)**: 1,204,741 params, 50 tokens.

**The 4→5 transition is NOT another step along the parameter axis.** Model 5 is only ~1.4× Model 4's
parameters (the tightest gap on the ladder), it **changes architecture family** (convolution → attention),
and it has **markedly lower accuracy** (0.6603 vs 0.7439). So a CMI difference here is better read as
**convolution-vs-attention, and possibly fit quality**, than as complexity. Settings otherwise identical to
Models 2–4 (DECISIONS_LOG: n_samples=8000 + pe 200, zero PM, predicted-class target, FA-CPU/IG-MPS/
KS-MPS-batched, N=500, 5 seeds); evidence in `notebooks/methodology/00_methodology_checks.ipynb`.

**No ground-truth recovery check** (no readable coefficients). **CMI with concentration as the confound
control is the headline.**

**Attention reduction (a methods decision, logged):** last-layer attention, mean over the 8 heads, mean over
queries = each region's *attention received*; one token ↔ one region (patch 60), no token→region aggregation.
Last-layer (not rollout) is deliberate: the thesis question is whether raw attention — what practitioners
read off a model — tracks feature importance (Jain & Wallace 2019); rollout would test a *repaired* version.
Rollout is a **pre-registered follow-up** if last-layer scores poorly. Attention is class-AGNOSTIC (one
ordering per sample, not per predicted class) and non-negative — noted where it matters.

**Run order (cheap first, KernelSHAP last).** §1–§9 run in ~36 min: FA (§3), IG (§4), Attention (§5, cheap),
saved (§6), their CMI (§7), concentration (§8), presentation (§9). Then start **§10 (KernelSHAP, ~1.5 h)**;
§10 saves per-seed. §11–§12 finish after; §12 is the four-rung ladder comparison.

> **You run the cells.** Every cell over ~1 min prints a time estimate up front.

## §1. Setup, settings, paths (all parameters defined once here)

In [ ]:
import sys, json, time
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import sleep_edf.config as cfg
from sleep_edf.loader import load_sleep_edf
from harness.models.transformer import build_transformer
from harness.models.cnn import torch_predict_proba                # black-box predict_proba wrapper (any torch model)
from harness.xai.regions import build_region_grid
from harness.xai.feature_ablation import feature_ablation
from harness.xai.kernel_shap import kernel_shap
from harness.xai.integrated_gradients import integrated_gradients
from harness.xai.deletion_curves import perturbation_curves
from harness.xai.cmi import compute_cmi
from harness.xai.concentration import region_concentration

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SEEDS       = [0, 1, 2, 3, 4]
PM          = "zero"
N_EVAL      = 500
EVAL_SEED   = 42
KS_N_SAMPLES = 8000
KS_PE        = 200
# target class = PREDICTED (harness default target_class=None). NB attention is class-AGNOSTIC (see §5).

MPS = "mps" if torch.backends.mps.is_available() else "cpu"
DEV_FA   = "cpu"     # FeatureAblation: tiny batch-1 forwards
DEV_IG   = MPS       # Integrated Gradients: batched fwd+bwd
DEV_ATT  = MPS       # Attention: one forward with need_weights (cheap)
DEV_KS   = MPS       # KernelSHAP batched
DEV_DEL  = "cpu"     # deletion curves / concentration
if MPS == "cpu": KS_PE = 1; print("!! MPS unavailable — KS falls back to CPU, perturbations_per_eval=1 (slower).")

CKPT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "checkpoints"
OUT_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "metrics";  OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "figures";  FIG_DIR.mkdir(parents=True, exist_ok=True)

grid = build_region_grid(cfg.INPUT_LENGTH, cfg.REGION_SIZE_PRIMARY_PCT)   # 3000 -> 50 regions of 60 (0.6 s each)
SEC_PER_REGION = int(np.unique(grid.sizes)[0]) / cfg.SAMPLING_RATE

def load_model5(seed, device):
    """Load Model 5 (transformer, PATCH 60 — the ladder rung) seed `seed`, eval mode, on `device`."""
    m = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                          n_classes=cfg.N_CLASSES, patch_size=cfg.TRANSFORMER_PATCH_SIZE)
    m.load_state_dict(torch.load(CKPT_DIR / f"model5_transformer_seed{seed}.pt", map_location="cpu")); m.eval()
    return m.to(device)

def stratified_idx(y, per_class, seed):
    rng = np.random.RandomState(seed); out = []
    for c in range(cfg.N_CLASSES): out += list(rng.choice(np.where(y == c)[0], per_class, replace=False))
    return np.array(out)

assert grid.n_regions == 50 and cfg.TRANSFORMER_PATCH_SIZE == 60, "attention needs 1 token per region (patch 60 -> 50 tokens)"
print(f"grid: {grid.n_regions} regions x {int(np.unique(grid.sizes)[0])} samples ({SEC_PER_REGION:.1f} s each) | "
      f"patch={cfg.TRANSFORMER_PATCH_SIZE} -> 50 tokens (1 per region) | MPS={MPS}")
print(f"settings: PM={PM} | target=PREDICTED | N_EVAL={N_EVAL} | seeds={SEEDS} | KS n_samples={KS_N_SAMPLES}, pe={KS_PE}")
print(f"device: FA={DEV_FA} IG={DEV_IG} ATT={DEV_ATT} KS={DEV_KS} deletion/concentration={DEV_DEL}")

## §2. Evaluation subset — LOAD the shared fixed subset (never regenerate)

The **same** `N = 500` fixed subset Models 2–4 used, LOADED from `results/metrics/xai_eval_subset_idx.npy`
(100/class, seed 42). Not rebuilt — raises if missing, so Model 5 runs on the identical 500 samples.

Also reports, per seed, how many of the 500 **Model 5** misclassifies and their breakdown. Model 5's accuracy
is markedly lower (bal. acc 0.6603 vs the CNNs' ~0.74), so expect **more** misclassifications — which matters
because the predicted-class rule means attribution follows the *predicted* class.

In [ ]:
EVAL_IDX_PATH = OUT_DIR / "xai_eval_subset_idx.npy"
X_test, y_test = load_sleep_edf("test", verbose=False)
if not EVAL_IDX_PATH.exists():
    raise FileNotFoundError(f"shared eval subset {EVAL_IDX_PATH} not found — run the Model 2 XAI notebook's "
                            "§2 first (it builds the ladder-wide subset). Model 5 must LOAD it, never rebuild.")
eval_idx = np.load(EVAL_IDX_PATH); print("loaded shared fixed eval subset:", EVAL_IDX_PATH.name)

eval_sigs = X_test[eval_idx].astype(float)            # (500, 3000) — SAME 500 samples as Models 2-4
eval_true = y_test[eval_idx]
assert len(eval_idx) == N_EVAL and set(np.bincount(eval_true)) == {N_EVAL // cfg.N_CLASSES}
print("per-class counts (true):", {CLASS_NAMES[c]: int((eval_true == c).sum()) for c in range(cfg.N_CLASSES)})

print(f"\n{'seed':>5}{'misclassified':>15}   breakdown (true->pred counts)")
for seed in SEEDS:
    pp = torch_predict_proba(load_model5(seed, DEV_FA), device=DEV_FA)
    pred = pp(eval_sigs).argmax(1)
    mis = int((pred != eval_true).sum()); br = {}
    for t, p in zip(eval_true[pred != eval_true], pred[pred != eval_true]):
        br[f"{CLASS_NAMES[t]}->{CLASS_NAMES[p]}"] = br.get(f"{CLASS_NAMES[t]}->{CLASS_NAMES[p]}", 0) + 1
    print(f"{seed:>5}{mis:>15}   {dict(sorted(br.items(), key=lambda kv: -kv[1])[:6])}")

## §3. FeatureAblation attributions — all 5 seeds  (~7 min, CPU)

Hides one region at a time (zero baseline), records the predicted-class probability drop = per-region
reliance. The validation control (near the oracle by construction). Result `fa_attr` (5, 500, 50).

In [ ]:
fa_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"FeatureAblation: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_FA}. Estimated ~7 min.", flush=True)
for si, seed in enumerate(SEEDS):
    pp = torch_predict_proba(load_model5(seed, DEV_FA), device=DEV_FA)
    t0 = time.perf_counter()
    for j in range(N_EVAL):
        fa_attr[si, j] = feature_ablation(pp, eval_sigs[j], grid, PM)      # target_class=None -> predicted
    dt = time.perf_counter() - t0
    print(f"  seed {seed}: {dt:.0f}s" + (f"  ->  est ~{dt*len(SEEDS)/60:.1f} min total" if si == 0 else ""), flush=True)
print("FeatureAblation done. fa_attr:", fa_attr.shape)

## §4. Integrated Gradients attributions — all 5 seeds  (~1.5 min, MPS)

IG integrates the model's gradient along a path from the zero baseline, summed per region (completeness).
Gradient-based, so it takes the torch model directly. Result `ig_attr` (5, 500, 50).

In [ ]:
ig_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"Integrated Gradients: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_IG}. Estimated ~1.5 min.", flush=True)
for si, seed in enumerate(SEEDS):
    model = load_model5(seed, DEV_IG)
    if si == 0:
        for _ in range(3): integrated_gradients(model, eval_sigs[0], grid, PM)   # warm MPS
    t0 = time.perf_counter()
    for j in range(N_EVAL):
        ig_attr[si, j] = integrated_gradients(model, eval_sigs[j], grid, PM)      # target None -> predicted
    dt = time.perf_counter() - t0
    print(f"  seed {seed}: {dt:.0f}s" + (f"  ->  est ~{dt*len(SEEDS)/60:.1f} min total" if si == 0 else ""), flush=True)
print("Integrated Gradients done. ig_attr:", ig_attr.shape)

## §5. Attention attributions — all 5 seeds  (~1 min, MPS)  ◀ transformer-only

The fourth method, and the reason patch 60 exists. `nn.TransformerEncoder` does not return attention weights,
so the cell **reproduces the encoder forward layer-by-layer** (respecting `norm_first`), calling
`self_attn(..., need_weights=True)` to capture per-layer, per-head attention (verified to match `model()`
logits exactly). The logged reduction: **last layer → mean over the 8 heads → mean over the 50 queries =
each region's "attention received"**. Patch 60 gives **one token per 60-sample region**, so these 50 token
scores *are* the 50 region scores — no aggregation.

Two properties to keep in mind: attention is **class-agnostic** (one ordering per sample regardless of the
predicted class — unlike FA/IG/KS) and **non-negative**. For CMI its single per-sample ordering drives the
deletion curves, which still track the predicted class. Result `att_attr` (5, 500, 50).

> **Heads-up (observed on the pilot checkpoint):** this transformer's attention **collapses to ~uniform in
> its deeper layers** (a known rank/entropy collapse in deep, underfit transformers) — last-layer attention
> received ≈ 1/50 for every region, i.e. **no per-region signal**. If so, the attention attribution is
> degenerate: undefined ordering → CMI ≈ 0 → rank-agreement undefined. That is a genuine result (raw
> last-layer attention carries no importance here — the extreme Jain & Wallace case), reported by the
> `std across regions` line below; it is **not** a bug. Rollout (earlier layers still have structure) is the
> pre-registered follow-up.

In [ ]:
def attention_received(model, sigs, device, bs=128):
    """Last-layer, mean-over-heads, mean-over-queries attention received per token (= region).
    Reproduces the transformer's encoder forward (norm_first) to capture the final layer's attention
    (need_weights=True). One token == one 60-sample region (patch 60), so output is (n, n_regions)."""
    dt = next(model.parameters()).dtype; model.eval()
    out = np.empty((len(sigs), grid.n_regions))
    for i in range(0, len(sigs), bs):
        Xb = torch.as_tensor(np.asarray(sigs[i:i + bs])[:, None, :], dtype=dt, device=device)
        with torch.no_grad():
            z = model.embed(Xb).transpose(1, 2) + model.pos          # (b, 50, d_model)
            last = None
            for layer in model.encoder.layers:                       # norm_first=True
                h = layer.norm1(z)
                ao, aw = layer.self_attn(h, h, h, need_weights=True, average_attn_weights=False)  # (b,H,Q,K)
                z = z + layer.dropout1(ao)
                z = z + layer._ff_block(layer.norm2(z))
                last = aw
            out[i:i + bs] = last.mean(1).mean(1).cpu().numpy()       # mean heads, mean queries -> (b, K=50)
    return out

att_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"Attention: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_ATT}. Estimated ~1 min.", flush=True)
for si, seed in enumerate(SEEDS):
    model = load_model5(seed, DEV_ATT)
    t0 = time.perf_counter()
    att_attr[si] = attention_received(model, eval_sigs, DEV_ATT)
    print(f"  seed {seed}: {time.perf_counter()-t0:.0f}s", flush=True)
# Distribution-over-regions sanity + DEGENERACY CHECK. Attention received sums to ~1 per sample. The key
# quantity is its STD across the 50 regions: on the pilot checkpoint the transformer's deeper-layer attention
# collapses to EXACTLY uniform (std ~0), so last-layer attention received = 1/50 everywhere and carries NO
# per-region signal -> its region ordering is undefined and its CMI ~0. That is a genuine result (the extreme
# "attention != feature importance" case), not a bug; rollout (earlier layers still have structure) is the
# pre-registered follow-up. This line makes the (non-)uniformity explicit.
att_std = float(np.nanmean(att_attr.std(axis=-1)))
print("attention done. att_attr:", att_attr.shape,
      "| per-sample sum ≈", round(float(np.nanmean(att_attr.sum(-1))), 3),
      "| mean std across regions:", round(att_std, 5))
if att_std < 1e-4:
    print("  >>> ATTENTION IS ~UNIFORM (std≈0): last-layer attention has no per-region signal for this model. "
          "Its CMI will be ~0 and its rank-agreement undefined -- a finding (raw last-layer attention carries "
          "no importance here), not a bug. Rollout is the pre-registered follow-up.")

## §6. Save §3 + §4 + §5 attributions to disk  (before KernelSHAP)

Persist FA, IG and Attention + the eval indices, so §7–§9 can run (and re-run) without recomputing and the
long §10 never risks the cheap results.

In [ ]:
np.savez(OUT_DIR / "model5_xai_fa_ig_att_attr.npz",
         fa_attr=fa_attr, ig_attr=ig_attr, att_attr=att_attr, eval_idx=eval_idx, seeds=np.array(SEEDS))
print("saved:", (OUT_DIR / "model5_xai_fa_ig_att_attr.npz").name)

## §7. CMI for FeatureAblation, IG and Attention  (~21 min, CPU)

Same mechanic as the CNN rungs. For each seed and sample, `perturbation_curves` orders the 50 regions by the
method's scores and builds MoRF (hide most-relevant first) and LeRF (least first) curves; `compute_cmi` →
DDS (front-weighted gap), PES (sign-consistency across samples), CMI (harmonic mean). Aggregated mean ± std
across seeds. Three methods here (KernelSHAP's CMI is §11).

In [ ]:
def cmi_for(attr, label, est_min):
    """Per-seed CMI/DDS/PES for an attribution array (n_seeds,N,50). Deletion curves on CPU (predicted class)."""
    print(f"{label}: deletion curves for {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_DEL}. Est ~{est_min} min.", flush=True)
    per_seed = []
    for si, seed in enumerate(SEEDS):
        pp = torch_predict_proba(load_model5(seed, DEV_DEL), device=DEV_DEL)
        M, L = [], []
        t0 = time.perf_counter()
        for j in range(N_EVAL):
            c = perturbation_curves(pp, eval_sigs[j], grid, attr[si, j], method=PM)   # target None -> predicted
            M.append(c["MoRF"]); L.append(c["LeRF"])
        r = compute_cmi(M, L)
        per_seed.append({"seed": seed, "CMI": r["CMI"], "DDS": r["DDS"], "PES": r["PES"]})
        if si == 0: print(f"  seed {seed}: {time.perf_counter()-t0:.0f}s -> est ~{(time.perf_counter()-t0)*len(SEEDS)/60:.1f} min", flush=True)
    return per_seed

def agg(per_seed, key):
    v = np.array([d[key] for d in per_seed]); return v.mean(), v.std()
def ms(per_seed, key):
    m, s = agg(per_seed, key); return f"{m:.3f}±{s:.3f}"

fa_cmi  = cmi_for(fa_attr,  "FeatureAblation",     7)
ig_cmi  = cmi_for(ig_attr,  "IntegratedGradients", 7)
att_cmi = cmi_for(att_attr, "Attention",           7)
for name, ps in [("FeatureAblation", fa_cmi), ("IntegratedGradients", ig_cmi), ("Attention", att_cmi)]:
    print(f"\n{name}:  CMI {ms(ps,'CMI')} | DDS {ms(ps,'DDS')} | PES {ms(ps,'PES')}")
    print("   per-seed PES:", [round(d["PES"], 3) for d in ps])

## §8. Concentration — the confound control  (~6 min, CPU)

Model property, not attribution: per sample, single-region deletions → per-region reliance → `1 − normalised
entropy` (0 = diffuse, 1 = concentrated). When CMI moves across the ladder, concentration says whether the
model's reliance profile changed shape (mechanical) or not (genuine). The convergence investigation measured
Model 5's concentration at ~0.092 on 10 samples — essentially Model 2's; confirm on the full N=500.

In [ ]:
print(f"Concentration: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_DEL}. Estimated ~6 min.", flush=True)
conc_per_seed = []
for si, seed in enumerate(SEEDS):
    pp = torch_predict_proba(load_model5(seed, DEV_DEL), device=DEV_DEL)
    t0 = time.perf_counter()
    vals = np.array([region_concentration(pp, eval_sigs[j], grid, PM) for j in range(N_EVAL)])
    conc_per_seed.append(float(np.nanmean(vals)))
    if si == 0: print(f"  seed {seed}: {time.perf_counter()-t0:.0f}s -> est ~{(time.perf_counter()-t0)*len(SEEDS)/60:.1f} min", flush=True)
concentration_mean, concentration_std = float(np.mean(conc_per_seed)), float(np.std(conc_per_seed))
print(f"\nconcentration: {concentration_mean:.3f} ± {concentration_std:.3f}   per-seed: {[round(c,3) for c in conc_per_seed]}")

## §9. Presentation — FA, IG, Attention (before KernelSHAP)

Attribution heatmaps over the 30-second epoch (per predicted class × region, mean over seeds), the
CMI/DDS/PES table for the three cheap methods, and their cross-method rank agreement. **Attention watch-thread:**
does attention agree with the perturbation/gradient methods, or is it off on its own (the time-series analogue
of Jain & Wallace)? Also — the transformer has **no GAP head** (it mean-pools tokens), so unlike the CNNs its
attributions *could* localise temporally within a class; note whether they do. (Interpret while §10 runs.)

In [ ]:
def per_class_heatmap(attr, seed_preds):
    H = np.full((cfg.N_CLASSES, grid.n_regions), np.nan)
    for c in range(cfg.N_CLASSES):
        rows = [attr[si][seed_preds[si] == c].mean(0) for si in range(len(SEEDS)) if (seed_preds[si] == c).any()]
        if rows: H[c] = np.mean(rows, axis=0)
    return H

seed_preds = np.array([torch_predict_proba(load_model5(s, DEV_FA), device=DEV_FA)(eval_sigs).argmax(1) for s in SEEDS])
extent = [0, grid.n_regions * SEC_PER_REGION, cfg.N_CLASSES - 0.5, -0.5]
cheap = [("FeatureAblation", fa_attr), ("IntegratedGradients", ig_attr), ("Attention", att_attr)]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (name, attr) in zip(ax, cheap):
    H = per_class_heatmap(attr, seed_preds); v = np.nanmax(np.abs(H))
    im = a.imshow(H, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, extent=extent)
    a.set_yticks(range(cfg.N_CLASSES)); a.set_yticklabels(CLASS_NAMES); a.set_xlabel("time within epoch (s)")
    a.set_title(f"{name} (attention is non-negative)" if name == "Attention" else name); fig.colorbar(im, ax=a, fraction=.04)
ax[0].set_ylabel("predicted stage")
fig.suptitle("Model 5 — per-region attribution over the 30 s epoch (mean over 5 seeds)", y=1.03)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_13_model5_xai_heatmaps_cheap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'method':<20}{'CMI':>16}{'DDS':>16}{'PES':>16}")
for name, ps in [("FeatureAblation", fa_cmi), ("IntegratedGradients", ig_cmi), ("Attention", att_cmi)]:
    print(f"{name:<20}{ms(ps,'CMI'):>16}{ms(ps,'DDS'):>16}{ms(ps,'PES'):>16}")

def pair_agree(A, B):
    """Mean per-sample Spearman over the 50 regions, across seeds. Skips samples where a vector is
    constant (spearman undefined -> nan), which happens when attention is near-uniform; also reports
    the fraction skipped, since heavy skipping is itself a finding (diffuse/uniform attention)."""
    vals = [spearmanr(A[si, j], B[si, j]).correlation
            for si in range(len(SEEDS)) for j in range(N_EVAL)]
    vals = np.array(vals, float); ok = ~np.isnan(vals)
    return (float(vals[ok].mean()) if ok.any() else float("nan")), float(1 - ok.mean())

print(f"\ncross-method rank agreement (mean Spearman; frac undefined = near-uniform samples):")
for a, b, lab in [(fa_attr, ig_attr, "FA-IG"), (fa_attr, att_attr, "FA-Att"), (ig_attr, att_attr, "IG-Att")]:
    r, skip = pair_agree(a, b); print(f"  {lab}: {r:.3f}" + (f"  ({skip:.0%} undefined)" if skip > 0.01 else ""))
print("  low FA-Att / IG-Att -> attention diverges from feature importance (the Jain & Wallace question).")

## §10. ▶ KernelSHAP attributions — all 5 seeds  (~1.5 hours, MPS, saves per seed)

**The long one — start it and walk away.** `n_samples=8000`, batched onto MPS (`perturbations_per_eval=200`).
Each seed **saved as it finishes** (`model5_xai_ks_attr_seed{seed}.npy`); interruption loses ≤1 seed, re-run
skips saved seeds. Expected ~18 min/seed → ~1.5 h (the transformer's batched KS is ~2.1 s/sample). **Note for
the dip-check (§11):** the methodology investigation found the transformer's KernelSHAP is systematically less
converged than the CNNs' (Spearman gap ~0.05, structural), so a LOW Model 5 KS CMI cannot be told apart from
under-sampling without the §11 numbers — a HIGH one is safe (under-sampling only deflates).

In [ ]:
ks_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"KernelSHAP: {len(SEEDS)} seeds x {N_EVAL} samples, n_samples={KS_N_SAMPLES}, pe={KS_PE}, on {DEV_KS}.")
print(f"Estimated ~1.5 hours total (~18 min/seed). Per-seed files are saved as each completes.", flush=True)
for si, seed in enumerate(SEEDS):
    seed_path = OUT_DIR / f"model5_xai_ks_attr_seed{seed}.npy"
    if seed_path.exists():
        ks_attr[si] = np.load(seed_path); print(f"  seed {seed}: loaded from disk ({seed_path.name})", flush=True); continue
    pp = torch_predict_proba(load_model5(seed, DEV_KS), device=DEV_KS)
    if si == 0:
        kernel_shap(pp, eval_sigs[0], grid, PM, n_samples=KS_N_SAMPLES, perturbations_per_eval=KS_PE)   # warm
    t0 = time.perf_counter()
    for j in range(N_EVAL):
        ks_attr[si, j] = kernel_shap(pp, eval_sigs[j], grid, PM, n_samples=KS_N_SAMPLES,
                                     target_class=None, seed=0, perturbations_per_eval=KS_PE)
    np.save(seed_path, ks_attr[si]); dt = time.perf_counter() - t0
    print(f"  seed {seed}: {dt/60:.1f} min" + (f" -> est ~{dt*len(SEEDS)/60:.0f} min total" if si == 0 else "") +
          f". Saved {seed_path.name}.", flush=True)
print("KernelSHAP done. ks_attr:", ks_attr.shape)

## §11. CMI for KernelSHAP + dip-check numbers  (~7 min)  ◀ the dip-check matters MOST here

KernelSHAP CMI (same mechanic as §7), then the **dip-check** (KS top-25 set overlap, top-10 order stability,
n_samples 4000 vs 8000 on a stratified subsample). **This rung is where the dip-check is load-bearing:**
Model 5's KernelSHAP is systematically less converged than the CNNs', so if Model 5's KS CMI comes in **below**
Model 4's in §12, it cannot be distinguished from an under-sampling artefact without these numbers — read them
against Model 4's recorded values, not buried. If Model 5's KS CMI is at/above Model 4's, the direction-of-bias
argument applies and it is safe (under-sampling only deflates CMI).

In [ ]:
ks_cmi = cmi_for(ks_attr, "KernelSHAP", 7)
print(f"\nKernelSHAP:  CMI {ms(ks_cmi,'CMI')} | DDS {ms(ks_cmi,'DDS')} | PES {ms(ks_cmi,'PES')}")

# Dip-check: KS stability 4000 vs 8000 on a 25-sample stratified subsample (seed 0 model).
sub = stratified_idx(eval_true, 5, EVAL_SEED)
pp0 = torch_predict_proba(load_model5(0, DEV_KS), device=DEV_KS)
a8 = ks_attr[0, sub]
print(f"dip-check: KS n=4000 on {len(sub)} samples (seed 0) for the 4000->8000 comparison ...", flush=True)
a4 = np.array([kernel_shap(pp0, eval_sigs[k], grid, PM, n_samples=4000, seed=0, perturbations_per_eval=KS_PE) for k in sub])
def topk(a, b, k):
    ta, tb = set(np.argsort(a)[::-1][:k]), set(np.argsort(b)[::-1][:k]); ix = list(tb)
    return len(ta & tb)/k, (spearmanr(a[ix], b[ix]).correlation if k > 1 else np.nan)
dip = {"full": float(np.mean([spearmanr(a4[i], a8[i]).correlation for i in range(len(sub))])),
       "ov25": float(np.mean([topk(a4[i], a8[i], 25)[0] for i in range(len(sub))])),
       "or10": float(np.mean([topk(a4[i], a8[i], 10)[1] for i in range(len(sub))]))}
print(f"KS 4000->8000 (n={len(sub)}): full Spearman {dip['full']:.3f} | top-25 overlap {dip['ov25']:.3f} | top-10 order {dip['or10']:.3f}")
print("Compare to Model 4's recorded dip-check IF Model 5's KS CMI is below Model 4's (§12).")

## §12. Full four-method presentation + Models 2–5 ladder comparison

All four methods together (FA, KernelSHAP, IG, **Attention**), then the **four-rung ladder comparison** (Models
2–5, read from saved results JSONs — nothing hardcoded), and the CMI-vs-concentration plot across all four
rungs. Watch-threads:

- **CMI direction / the 4→5 read.** 2→3→4 rose; Model 5 is a different architecture family at ~1.4× params and
  lower accuracy, so read a 4→5 CMI change as **convolution-vs-attention / fit quality**, not complexity.
- **PES ceiling** (corrected flag): report SATURATED (≥~0.97) vs a genuine drop — if pinned, CMI ≈ DDS here.
- **Attention vs the others** — attention's CMI and its rank agreement with FA/KS/IG. Low agreement **and** low
  CMI would be the time-series analogue of Jain & Wallace (attention ≠ feature importance).
- **Dip check** — if Model 5's KS CMI is below Model 4's, read §11's stability numbers before calling it a
  genuine dip (under-sampling deflates CMI, and Model 5's KS is the least converged rung).
- **Heatmaps** — did the transformer (no GAP head, mean-pool) localise attribution temporally where the CNNs
  did not?

In [ ]:
methods = [("FeatureAblation", fa_attr, fa_cmi), ("KernelSHAP", ks_attr, ks_cmi),
           ("IntegratedGradients", ig_attr, ig_cmi), ("Attention", att_attr, att_cmi)]

# CMI/DDS/PES table (4 methods) + corrected PES flag
print(f"{'method':<20}{'CMI':>16}{'DDS':>16}{'PES (mean±std)':>18}   per-seed PES")
for name, _, ps in methods:
    print(f"{name:<20}{ms(ps,'CMI'):>16}{ms(ps,'DDS'):>16}{ms(ps,'PES'):>18}   {[round(d['PES'],3) for d in ps]}")
print(f"\nconcentration (confound control): {concentration_mean:.3f} ± {concentration_std:.3f}")
# PES watch-thread -- distinguish SATURATION (at the ceiling) from a GENUINE drop, not "any value < 1.0".
pes_vals = [d["PES"] for _, _, ps in methods for d in ps]
pes_min = min(pes_vals)
if pes_min >= 0.97:
    print(f"PES watch-thread: SATURATED (all method/seed PES >= 0.97; min {pes_min:.3f}, most = 1.000). "
          "CMI is the harmonic mean of DDS and PES, so with PES at the ceiling CMI carries no information "
          "beyond DDS -- the consistency component is NOT discriminating in this setting.")
elif pes_min < 0.9:
    print(f"PES watch-thread: GENUINE DROP -- PES falls to {pes_min:.3f} (< 0.9) for some method/seed, so "
          "the consistency component IS discriminating here. Report which method/seed.")
else:
    print(f"PES watch-thread: near-ceiling (min {pes_min:.3f}, in [0.9, 0.97)) -- mostly saturated with some "
          "single-sample variation, not a break from the ceiling; CMI still tracks DDS closely.")

# Four attribution heatmaps
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for a, (name, attr, _) in zip(ax, methods):
    H = per_class_heatmap(attr, seed_preds); v = np.nanmax(np.abs(H))
    im = a.imshow(H, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, extent=extent)
    a.set_yticks(range(cfg.N_CLASSES)); a.set_yticklabels(CLASS_NAMES); a.set_xlabel("time (s)"); a.set_title(name)
    fig.colorbar(im, ax=a, fraction=.04)
ax[0].set_ylabel("predicted stage")
fig.suptitle("Model 5 — attribution over the epoch, all four methods (mean over 5 seeds)", y=1.03)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_13_model5_xai_heatmaps_all.png", dpi=150, bbox_inches="tight")
plt.show()

# Six pairwise cross-method agreements (attention pairs surfaced explicitly)
A = {"FA": fa_attr, "KS": ks_attr, "IG": ig_attr, "Att": att_attr}
raw_agree = {f"{a}-{b}": pair_agree(A[a], A[b]) for a, b in [("FA","KS"),("FA","IG"),("FA","Att"),("KS","IG"),("KS","Att"),("IG","Att")]}
agreement = {k: v[0] for k, v in raw_agree.items()}      # value only, for the JSON + ladder comparison
print("\ncross-method rank agreement (mean Spearman; undef = near-uniform-attention samples skipped):")
print("  " + " | ".join(f"{k} {v[0]:.3f}" + (f"({v[1]:.0%}u)" if v[1] > 0.01 else "") for k, v in raw_agree.items()))
print("  ATTENTION pairs: FA-Att, KS-Att, IG-Att -> low = attention diverges from importance (Jain & Wallace).")

results = {"model": "model5_transformer", "n_eval": N_EVAL, "eval_seed": EVAL_SEED, "pm": PM,
           "ks_n_samples": KS_N_SAMPLES, "concentration": {"mean": concentration_mean, "std": concentration_std},
           "agreement": agreement, "ks_dip_check": dip,
           "methods": {name: {"per_seed": ps, "CMI_mean": agg(ps,"CMI")[0], "CMI_std": agg(ps,"CMI")[1],
                              "DDS_mean": agg(ps,"DDS")[0], "PES_mean": agg(ps,"PES")[0]}
                       for name, _, ps in methods}}
json.dump(results, open(OUT_DIR / "model5_xai_cmi_results.json", "w"), indent=2, default=float)
print("\nsaved:", (OUT_DIR / "model5_xai_cmi_results.json").name)

### Ladder comparison — Models 2, 3, 4, 5

Four rungs side by side: CMI / DDS / PES per method (the three methods all rungs share) + concentration, with
per-step deltas; Models 2–4 read from their saved JSONs (missing → skipped). **Attention** is Model-5-only, so
its CMI/DDS/PES are reported separately (no cross-rung delta). Then the CMI-vs-concentration plot across the
rungs. Read the 4→5 step as architecture-family (conv→attention) + fit quality, not complexity.

In [ ]:
PREV = [("Model 2","model2_xai_cmi_results.json","8,181"), ("Model 3","model3_xai_cmi_results.json","93,285"),
        ("Model 4","model4_xai_cmi_results.json","863,557")]
SHARED = ["FeatureAblation", "KernelSHAP", "IntegratedGradients"]     # methods all rungs share

rungs = []
for label, fname, params in PREV:
    p = OUT_DIR / fname
    if p.exists(): rungs.append((label, params, json.load(open(p))))
    else: print(f"[skip] {label} results JSON ({fname}) not found — run its final section first.")
cur = {"concentration": {"mean": concentration_mean}, "agreement": agreement,
       "methods": {n: {"CMI_mean": agg(ps,"CMI")[0], "DDS_mean": agg(ps,"DDS")[0], "PES_mean": agg(ps,"PES")[0]}
                   for n, _, ps in methods}}
rungs.append(("Model 5", "1,204,741", cur)); labels = [r[0] for r in rungs]

for key in ["CMI", "DDS", "PES"]:
    print(f"\n{key} across rungs ({' -> '.join(labels)}):")
    print(f"{'method':<20}" + "".join(f"{l:>10}" for l in labels) + f"{'Δ(last step)':>14}")
    for name in SHARED:
        vals = [r[2]["methods"][name][f"{key}_mean"] for r in rungs]
        step = (vals[-1] - vals[-2]) if len(vals) > 1 else float("nan")
        print(f"{name:<20}" + "".join(f"{v:>10.3f}" for v in vals) + f"{step:>+14.3f}")

# Attention (Model 5 only)
print(f"\nAttention (Model 5 only):  CMI {ms(att_cmi,'CMI')} | DDS {ms(att_cmi,'DDS')} | PES {ms(att_cmi,'PES')}")

conc = [r[2]["concentration"]["mean"] for r in rungs]
if len(conc) > 1:
    print(f"\n{'concentration':<20}" + "".join(f"{c:>10.3f}" for c in conc) + f"{conc[-1]-conc[-2]:>+14.3f}")

# CMI-vs-concentration plot across the rungs (KernelSHAP through-line)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2)); x = range(len(labels))
for name, mk in [("FeatureAblation","o-"),("KernelSHAP","s-"),("IntegratedGradients","^-")]:
    ax[0].plot(x, [r[2]["methods"][name]["CMI_mean"] for r in rungs], mk, label=name)
ax[0].plot(x, conc, "d--", color="grey", label="concentration")
ax[0].set_xticks(list(x)); ax[0].set_xticklabels(labels); ax[0].set_ylabel("CMI / concentration")
ax[0].set_title("CMI (shared methods) + concentration across rungs"); ax[0].legend(fontsize=8)
ks_vals = [r[2]["methods"]["KernelSHAP"]["CMI_mean"] for r in rungs]
ax[1].plot(conc, ks_vals, "s-")
for xi, l in zip(range(len(labels)), labels): ax[1].annotate(l, (conc[xi], ks_vals[xi]), fontsize=8)
ax[1].set_xlabel("concentration (confound)"); ax[1].set_ylabel("KernelSHAP CMI")
ax[1].set_title("KS CMI vs concentration across the ladder")
fig.suptitle("Ladder faithfulness trend vs the concentration confound (Models 2-5)", y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_13_model5_xai_cmi_vs_concentration.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_13_model5_xai_cmi_vs_concentration.png").name)

### Verdict (fill in after running)

- **The ladder, complete.** CMI trend Models 2→5 per method, with the 4→5 step read as **conv-vs-attention +
  fit quality** (different family, ~1.4× params, lower accuracy), not another complexity step. Does the
  monotonic-degradation hypothesis hold, or did faithfulness rise / stay flat / dip?
- **Attention.** Its CMI, and its agreement with FA/KS/IG. Low agreement + low CMI = the time-series analogue
  of attention ≠ feature importance (Jain & Wallace). Note that attention is class-agnostic and non-negative.
- **Dip check.** If Model 5's KS CMI fell below Model 4's, are §11's stability numbers comparable to Model
  4's, or is the drop an under-sampling artefact (Model 5's KS is the least-converged rung)?
- **PES ceiling.** SATURATED or a genuine drop? If saturated, CMI ≈ DDS across the whole ladder — a metric
  finding for the discussion.
- **Concentration** next to CMI (and the plot): is any CMI change genuine or tracking the confound?
- **Heatmaps.** Did the transformer localise attribution temporally (no GAP head), unlike the CNNs?